# Database Schema Explorer
Shows exactly how storm data is stored in PostgreSQL — tables, columns, types, sample rows, and relationships.

In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import psycopg2
from dotenv import load_dotenv

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

assert "DATABASE_URL" in os.environ, "DATABASE_URL not found — add it to .env at the project root."

conn = psycopg2.connect(os.environ["DATABASE_URL"])
print("Connected to DB successfully.")

Connected to DB successfully.


## 1 · Tables in the database

In [3]:
tables = pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""", conn)

print("Tables in storm_db:")
tables

Tables in storm_db:


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\4086693071.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables = pd.read_sql("""


,table_name
0,geography_columns
1,geometry_columns
2,nature_types
3,spatial_ref_sys
4,storm_observations
5,storms


## 2 · Column definitions for each table

In [4]:
for table in ["nature_types", "storms", "storm_observations"]:
    cols = pd.read_sql(f"""
        SELECT
            column_name,
            data_type,
            is_nullable,
            column_default
        FROM information_schema.columns
        WHERE table_name = '{table}'
        ORDER BY ordinal_position
    """, conn)
    print(f"\n{'='*60}")
    print(f"  TABLE: {table}")
    print(f"{'='*60}")
    print(cols.to_string(index=False))


  TABLE: nature_types
column_name data_type is_nullable column_default
         id  smallint          NO           None
       code character          NO           None
description      text          NO           None

  TABLE: storms
column_name data_type is_nullable column_default
    atcf_id      text          NO           None
     season  smallint          NO           None
      basin character          NO           None
   subbasin character         YES           None

  TABLE: storm_observations
column_name                data_type is_nullable                                 column_default
         id                   bigint          NO nextval('storm_observations_id_seq'::regclass)
    atcf_id                     text          NO                                           None
   iso_time timestamp with time zone          NO                                           None
        lat                     real          NO                                           None
        lo

C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\1830393844.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cols = pd.read_sql(f"""
C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\1830393844.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cols = pd.read_sql(f"""
C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\1830393844.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cols = pd.read_sql(f"""


## 3 · `nature_types` — lookup table
Maps integer codes to storm nature labels. The `storm_observations` table references this via FK.

In [5]:
nature = pd.read_sql("SELECT * FROM nature_types ORDER BY id", conn)
nature

C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\169111031.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  nature = pd.read_sql("SELECT * FROM nature_types ORDER BY id", conn)


,id,code,description
0,0,DS,Disturbance
1,1,ET,Extratropical
2,2,MX,Mixed/Subtropical
3,3,SS,Subtropical
4,4,TS,Tropical


## 4 · `storms` — one row per storm
Each storm has a unique ATCF ID (e.g. `AL012005`), a season year, basin, and subbasin.

In [7]:
storms_sample = pd.read_sql("""
    SELECT * FROM storms
    ORDER BY season DESC, atcf_id
    LIMIT 10
""", conn)

print("Sample rows (10 most recent storms):")
storms_sample

Sample rows (10 most recent storms):


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\774268566.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  storms_sample = pd.read_sql("""


,atcf_id,season,basin,subbasin
0,WP012026,2026,WP,MM
1,WP022026,2026,WP,MM
2,WP012025,2025,WP,MM
3,WP022025,2025,WP,MM
4,WP032025,2025,WP,MM
5,WP042025,2025,WP,MM
6,WP052025,2025,WP,MM
7,WP062025,2025,WP,MM
8,WP072025,2025,WP,MM
9,WP082025,2025,WP,MM


In [8]:
stats = pd.read_sql("""
    SELECT
        COUNT(*)                   AS total_storms,
        MIN(season)                AS earliest_season,
        MAX(season)                AS latest_season,
        COUNT(DISTINCT basin)      AS basins,
        COUNT(DISTINCT subbasin)   AS subbasins
    FROM storms
""", conn)

print("storms table stats:")
stats

storms table stats:


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\2839684324.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stats = pd.read_sql("""


,total_storms,earliest_season,latest_season,basins,subbasins
0,2343,1945,2026,3,3


In [9]:
basins = pd.read_sql("""
    SELECT basin, COUNT(*) AS storm_count
    FROM storms
    GROUP BY basin
    ORDER BY storm_count DESC
""", conn)

print("Storms per basin:")
basins

Storms per basin:


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\2737656326.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  basins = pd.read_sql("""


,basin,storm_count
0,WP,2302
1,EP,36
2,NI,5


## 5 · `storm_observations` — one row per storm × 3h timestep
This is the main table the ML pipeline reads from. Each row is one observation of a storm.

In [10]:
obs_sample = pd.read_sql("""
    SELECT
        o.id, o.atcf_id, o.iso_time,
        o.lat, o.lon,
        n.code AS nature,
        o.dist2land, o.landfall,
        o.wind_speed, o.storm_pres,
        o.usa_sshs, o.usa_poci, o.usa_roci, o.usa_rmw,
        o.storm_speed, o.storm_dir
    FROM storm_observations o
    JOIN nature_types n ON o.nature = n.id
    WHERE o.atcf_id = (
        SELECT atcf_id FROM storms WHERE season = 2005 LIMIT 1
    )
    ORDER BY o.iso_time
    LIMIT 12
""", conn)

print("12 observations for one storm (one row = 3h timestep):")
obs_sample

12 observations for one storm (one row = 3h timestep):


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\2097062174.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  obs_sample = pd.read_sql("""


,id,atcf_id,iso_time,lat,lon,nature,dist2land,landfall,wind_speed,storm_pres,usa_sshs,usa_poci,usa_roci,usa_rmw,storm_speed,storm_dir
0,30046,WP012005,2005-01-13 00:00:00+00:00,5.2,153.1,DS,918.0,907.0,15.0,1006.0,-3,1008.7500,191.33333,37.285713,9.0,270.0
1,30047,WP012005,2005-01-13 03:00:00+00:00,5.2,152.6,DS,905.0,895.0,15.0,1006.0,-3,1008.7500,191.33333,37.285713,10.0,270.0
2,30048,WP012005,2005-01-13 06:00:00+00:00,5.2,152.1,DS,895.0,887.0,15.0,1006.0,-3,1008.7500,191.33333,37.285713,12.0,270.0
3,30049,WP012005,2005-01-13 09:00:00+00:00,5.2,151.4,DS,887.0,885.0,18.0,1005.0,-3,1008.7500,191.33333,37.285713,14.0,270.0
4,30050,WP012005,2005-01-13 12:00:00+00:00,5.2,150.7,DS,885.0,875.0,20.0,1004.0,-3,1006.0000,120.00000,50.000000,12.0,265.0
5,30051,WP012005,2005-01-13 15:00:00+00:00,5.1,150.2,DS,864.0,846.0,23.0,1003.0,-3,1008.7500,191.33333,37.285713,10.0,265.0
6,30052,WP012005,2005-01-13 18:00:00+00:00,5.1,149.7,TS,846.0,834.0,25.0,1002.0,-1,1005.6667,244.16667,48.761906,9.0,265.0
7,30053,WP012005,2005-01-13 21:00:00+00:00,5.1,149.2,TS,831.0,828.0,25.0,1002.0,-1,1005.6667,244.16667,48.761906,8.0,280.0
8,30054,WP012005,2005-01-14 00:00:00+00:00,5.3,148.9,TS,845.0,841.0,25.0,1004.0,-1,1005.6667,244.16667,48.761906,9.0,290.0
9,30055,WP012005,2005-01-14 03:00:00+00:00,5.4,148.4,TS,847.0,846.0,28.0,1003.0,-1,1005.6667,244.16667,48.761906,10.0,290.0


In [11]:
obs_stats = pd.read_sql("""
    SELECT
        COUNT(*)                        AS total_observations,
        COUNT(DISTINCT atcf_id)         AS unique_storms,
        MIN(iso_time)                   AS earliest,
        MAX(iso_time)                   AS latest,
        ROUND(AVG(wind_speed)::numeric, 1) AS avg_wind_knots,
        ROUND(MIN(wind_speed)::numeric, 1) AS min_wind,
        ROUND(MAX(wind_speed)::numeric, 1) AS max_wind,
        ROUND(AVG(storm_pres)::numeric, 1) AS avg_pressure_mb
    FROM storm_observations
""", conn)

print("storm_observations table stats:")
obs_stats

C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\511714046.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  obs_stats = pd.read_sql("""


storm_observations table stats:


,total_observations,unique_storms,earliest,latest,avg_wind_knots,min_wind,max_wind,avg_pressure_mb
0,132646,2343,1945-04-19 12:00:00+00:00,2026-02-06 12:00:00+00:00,53.0,-99.0,185.0,983.4


## 6 · What each column means

| Column | Type | Description |
|---|---|---|
| `id` | BIGSERIAL | Auto-increment PK |
| `atcf_id` | TEXT | Storm identifier (e.g. `AL062023`) — FK → `storms` |
| `iso_time` | TIMESTAMPTZ | Observation timestamp, every 3h |
| `lat` | REAL | Latitude (°N) |
| `lon` | REAL | Longitude (°E) |
| `geom` | GEOGRAPHY | PostGIS point (auto-generated from lat/lon) |
| `nature` | SMALLINT | Storm type code — FK → `nature_types` |
| `dist2land` | REAL | Distance to nearest land (km) |
| `landfall` | REAL | 1 if making landfall, 0 otherwise |
| `wind_speed` | REAL | Maximum sustained wind speed (knots) |
| `storm_pres` | REAL | Minimum central pressure (mb) |
| `usa_sshs` | SMALLINT | Saffir-Simpson Hurricane Wind Scale category (-5 to 5) |
| `usa_poci` | REAL | Pressure of outermost closed isobar (mb) |
| `usa_roci` | REAL | Radius of outermost closed isobar (km) |
| `usa_rmw` | REAL | Radius of maximum winds (km) |
| `storm_speed` | REAL | Storm translation speed (knots) |
| `storm_dir` | REAL | Storm heading direction (degrees, 0–360) |

## 7 · PostGIS geometry column
`geom` is auto-generated from `lat`/`lon`. You can use it for spatial queries, e.g. finding all storms that passed within 100 km of a point.

In [12]:
geom_sample = pd.read_sql("""
    SELECT
        atcf_id,
        iso_time,
        lat, lon,
        ST_AsText(geom)  AS geom_wkt,
        ST_AsGeoJSON(geom)::json AS geom_geojson
    FROM storm_observations
    LIMIT 5
""", conn)

print("geom column — WKT and GeoJSON representations:")
geom_sample[["atcf_id", "iso_time", "lat", "lon", "geom_wkt"]]

C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\3884485471.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  geom_sample = pd.read_sql("""


geom column — WKT and GeoJSON representations:


,atcf_id,iso_time,lat,lon,geom_wkt
0,WP012026,2026-01-14 18:00:00+00:00,9.5,130.2,POINT(130.1999969482422 9.5)
1,WP012026,2026-01-14 21:00:00+00:00,9.3,130.0,POINT(130 9.300000190734863)
2,WP012026,2026-01-15 00:00:00+00:00,9.3,129.7,POINT(129.6999969482422 9.300000190734863)
3,WP012026,2026-01-15 03:00:00+00:00,9.6,129.3,POINT(129.3000030517578 9.600000381469727)
4,WP012026,2026-01-15 06:00:00+00:00,10.0,128.9,POINT(128.89999389648438 10)


## 8 · Observations per storm (distribution)
Shows how many 3h timesteps each storm has — important for understanding how many sliding windows each storm contributes to training.

In [13]:
obs_per_storm = pd.read_sql("""
    SELECT
        o.atcf_id,
        s.season,
        s.basin,
        COUNT(*) AS num_observations,
        COUNT(*) * 3 AS duration_hours,
        ROUND(MAX(o.wind_speed)::numeric, 0) AS peak_wind_knots
    FROM storm_observations o
    JOIN storms s ON o.atcf_id = s.atcf_id
    GROUP BY o.atcf_id, s.season, s.basin
    ORDER BY num_observations DESC
    LIMIT 10
""", conn)

print("Top 10 longest-tracked storms:")
obs_per_storm

Top 10 longest-tracked storms:


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\3068658575.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  obs_per_storm = pd.read_sql("""


,atcf_id,season,basin,num_observations,duration_hours,peak_wind_knots
0,WP291990,1990,WP,208,624,140.0
1,WP081972,1972,WP,198,594,145.0
2,WP191996,1996,WP,175,525,115.0
3,WP331994,1994,WP,173,519,115.0
4,WP131986,1986,WP,170,510,90.0
5,WP301990,1990,EP,165,495,140.0
6,WP121997,1997,WP,162,486,90.0
7,WP371996,1996,WP,162,486,50.0
8,WP131996,1996,WP,159,477,95.0
9,WP302013,2013,WP,159,477,35.0


In [14]:
dist = pd.read_sql("""
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY cnt) AS p25_obs,
        percentile_cont(0.50) WITHIN GROUP (ORDER BY cnt) AS median_obs,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY cnt) AS p75_obs,
        AVG(cnt)                                           AS mean_obs,
        MIN(cnt)                                           AS min_obs,
        MAX(cnt)                                           AS max_obs
    FROM (
        SELECT atcf_id, COUNT(*) AS cnt
        FROM storm_observations
        GROUP BY atcf_id
    ) t
""", conn)

print("Distribution of observations per storm:")
dist

Distribution of observations per storm:


C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\2569023466.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dist = pd.read_sql("""


,p25_obs,median_obs,p75_obs,mean_obs,min_obs,max_obs
0,34.0,54.0,74.0,56.613743,1,208


## 9 · ML split sizes (season-based)
Reproduces the train/val/test split used in the ML pipeline without loading PyTorch.

In [15]:
splits = pd.read_sql("""
    SELECT
        CASE
            WHEN s.season <= 2014 THEN 'train (≤2014)'
            WHEN s.season BETWEEN 2015 AND 2019 THEN 'val (2015–2019)'
            ELSE 'test (≥2020)'
        END AS split,
        COUNT(DISTINCT s.atcf_id) AS storms,
        COUNT(o.id)               AS observations
    FROM storms s
    JOIN storm_observations o ON s.atcf_id = o.atcf_id
    GROUP BY split
    ORDER BY MIN(s.season)
""", conn)

print("Data split breakdown:")
splits

C:\Users\Admin\AppData\Local\Temp\ipykernel_22340\3798465816.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  splits = pd.read_sql("""


Data split breakdown:


,split,storms,observations
0,train (≤2014),2019,115291
1,val (2015–2019),158,8825
2,test (≥2020),166,8530


In [16]:
conn.close()
print("Done. Connection closed.")

Done. Connection closed.
